# test_client.py — API 서버 테스트용 클라이언트

## 이 파일은 무엇인가요?

이 파일은 `main.py`로 구동되는 FastAPI 서버가 **정상적으로 동작하는지 테스트**하기 위한 스크립트입니다.

**실제 서비스에서는 Spring Boot 같은 메인 서버**가 이 FastAPI 서버에 요청을 보내지만, 개발 단계에서는 이 스크립트를 직접 실행하여 빠르게 테스트할 수 있습니다.

### 사용 방법
1. **먼저** 터미널에서 FastAPI 서버를 실행합니다: `uvicorn main:app --host 0.0.0.0 --port 8080`
2. **다음** 별도의 터미널에서 이 스크립트를 실행합니다: `python test_client.py`

### 동작 원리
이 스크립트는 `requests` 라이브러리를 사용하여 로컬에서 실행 중인 FastAPI 서버(`localhost:8080`)에 HTTP POST 요청을 보냅니다.

요청에는 다음 3가지 정보가 JSON 형태로 담깁니다:
- **user_id**: 테스트용 사용자 ID
- **pet_image_url**: 인터넷에 공개된 강아지 사진 URL (Unsplash 무료 이미지)
- **cloth_image_url**: 인터넷에 공개된 옷 사진 URL (Unsplash 무료 이미지)

서버는 이 URL에서 이미지를 다운로드하고, AI 파이프라인을 돌린 후, 결과를 JSON으로 응답합니다.

## 1단계: 라이브러리 불러오기

- `requests`: HTTP 요청(GET, POST 등)을 아주 쉽게 보낼 수 있는 파이썬 라이브러리입니다.
  - `urllib`보다 훨씬 사용이 간편하여, API 테스트나 웹 크롤링에 가장 많이 사용됩니다.

In [ ]:
import requests

## 2단계: 요청 URL 및 테스트 데이터 설정

### 서버 주소
- `http://localhost:8080`: 내 컴퓨터(로컬)에서 8080번 포트로 실행 중인 FastAPI 서버를 가리킵니다.
- `/api/v1/fit`: `main.py`에서 `@app.post("/api/v1/fit")`로 정의한 엔드포인트(API 경로)입니다.

### 테스트 데이터 (payload)
실제 서비스에서는 사용자가 업로드한 사진의 URL이 전달되지만, 테스트에서는 Unsplash(무료 고화질 사진 사이트)의 공개 이미지 URL을 사용합니다.

| 필드 | 값 | 설명 |
|---|---|---|
| `user_id` | `"test_user_001"` | 테스트용 가짜 사용자 ID |
| `pet_image_url` | Unsplash 강아지 사진 URL | 테스트용 반려동물 사진 |
| `cloth_image_url` | Unsplash 옷 사진 URL | 테스트용 옷 사진 |

In [ ]:
url = "http://localhost:8080/api/v1/fit"

payload = {
    "user_id": "test_user_001",
    "pet_image_url": "https://images.unsplash.com/photo-1543466835-00a7907e9de1?q=80&w=500",
    "cloth_image_url": "https://images.unsplash.com/photo-1576566588028-4147f3842f27?q=80&w=500"
}

## 3단계: 서버에 POST 요청 보내기 및 결과 확인

- `requests.post(url, json=payload)`: 지정한 URL에 JSON 데이터를 담아 HTTP POST 요청을 보냅니다.
  - `json=payload`를 사용하면 `Content-Type: application/json` 헤더가 자동으로 설정됩니다.
- AI 파이프라인 연산이 무거우므로 **응답이 오기까지 1~2분 이상 대기**할 수 있습니다.
- `response.status_code`: HTTP 응답 코드 (200=성공, 500=서버 에러 등)
- `response.json()`: 서버가 반환한 JSON 응답 본문

### 정상 응답 시 출력 예시
```
응답 상태 코드: 200
결과 데이터: {'status': 'success', 'message': 'Fitting completed successfully.', 'result_url': 'https://storage.googleapis.com/...'}
```

### 에러 발생 시 출력 예시
```
응답 상태 코드: 500
결과 데이터: {'detail': '에러 메시지 내용...'}
```

In [ ]:
print("서버로 가상 피팅 요청을 보냅니다... (1~2분 대기)")
response = requests.post(url, json=payload)

print("응답 상태 코드:", response.status_code)
print("결과 데이터:", response.json())